# 00 — Setup and data
Run this notebook first. No model is trained here.

**Colab:** extract the complete ZIP to `/content/lid_finetuning_bundle` before running. **Local:** use Python 3.10–3.12 and a C++ compiler (WSL2 on Windows).


In [1]:
from pathlib import Path
import os, sys
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/lid_finetuning_bundle')]
ROOT = next((p for p in candidates if (p/'config.json').exists() and (p/'lidlab').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Extract the complete ZIP first and set ROOT to its lid_finetuning_bundle folder.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Project folder:', ROOT)


Project folder: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\new_method


## 1. Install packages and build the continued-training program

In [2]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
from scripts.build_native import build
print(build(ROOT))

d:\Projects\ML Projects\LangID - DSE project\data_pipeline\new_method\bin\lid_native


## 2. Put the supplied files in `data/`

```text
data/
├── train.csv
├── val.csv
├── test.csv
├── train_mixed_11groups.jsonl
├── val_mixed_11groups.jsonl
├── dataset_11groups_report.json
└── preprocessed/
    ├── commonlid.jsonl
    ├── flores_plus.jsonl
    └── wili-2018.jsonl
```

The two mixed JSONL files already contain the three target languages plus the other eight. Do not concatenate the target CSVs with them again. The report JSON is documentation, not training data.

The three files under `preprocessed/` are the evaluation benchmarks. Keep them separate; every experiment is evaluated on all three.

Both training arms start from the original pretrained model by default. Set `replay_start` to `target_only` only for sequential recovery, and use a new output directory.

Fix hyperparameters using validation data before the final test run. Defaults are starting settings, not optimized values.

In [3]:
import json
config = json.loads((ROOT/'config.json').read_text())
print(json.dumps(config, indent=2))

{
  "data": {
    "train": "data/train.csv",
    "validation": "data/val.csv",
    "target_test": "data/test.csv",
    "mixed_train": "data/train_mixed_11groups.jsonl",
    "mixed_validation": "data/val_mixed_11groups.jsonl",
    "benchmarks": {
      "commonlid": "data/preprocessed/commonlid.jsonl",
      "flores_plus": "data/preprocessed/flores_plus.jsonl",
      "wili_2018": "data/preprocessed/wili-2018.jsonl"
    },
    "dataset_report": "data/dataset_11groups_report.json"
  },
  "columns": {
    "text": "text",
    "label": "label",
    "group": "group",
    "doc_id": "group_id"
  },
  "normalization": "NFC",
  "require_document_ids": false,
  "strict_script_check": false,
  "label_aliases": {},
  "output_dir": "results/main_seed42",
  "seed": 42,
  "target_passes": 3,
  "replay_start": "base",
  "device": "auto",
  "conlid_batch_size": 32,
  "models": {
    "nllb": {
      "repo_id": "facebook/fasttext-language-identification",
      "filename": "model.bin",
      "revision": nul

In [4]:
# Optional edits. Uncomment only the settings you need.
# config['columns']['text'] = 'sentence'
# config['columns']['label'] = 'language'
# config['columns']['group'] = 'language_script_group'
# config['columns']['doc_id'] = 'group_id'
# config['label_aliases'] = {'0':'sin_Sinh', '1':'pli_Sinh', '2':'san_Sinh'}  # ONLY if this is your real mapping!
# config['device'] = 'cpu'
# config['replay_start'] = 'target_only'
# config['output_dir'] = 'results/sequential_seed42'
# config['require_document_ids'] = True  # enable after rebuilding fully group-disjoint splits
(ROOT/'config.json').write_text(json.dumps(config, indent=2), encoding='utf8')

1467

## 3. Audit the data
This checks expected categories, empty rows, cross-split text overlap, script presence for the target categories, and document overlap when IDs exist. Repeated benchmark texts with the same label are kept once. If one text has conflicting labels, every occurrence is removed because its ground truth is ambiguous. All removals are recorded. It does not replace your near-duplicate or annotation-quality checks.

The supplied target files share one `group_id` across their splits, so strict group-disjoint enforcement is disabled by default. This is recorded in the audit. Every model uses the same NFC/whitespace preprocessing.

In [5]:
from lidlab.data import load_config, prepare
c = load_config()
data = prepare(c)
import pandas as pd
counts = pd.DataFrame({k: data[k].label.value_counts() for k in ['train','validation','target_test','mixed_train','mixed_validation']}).fillna(0).astype(int)
display(counts)
for name,frame in data['benchmarks'].items():
    index_path = ROOT/c['output_dir']/f'evaluation_index_{name}.csv'
    frame[['sample_id','text_sha256','label']].to_csv(index_path, index=False)
    print(name, len(frame), 'selected rows; index:', index_path)
print('Data audit passed. Continue with notebooks 01–03.')

,train,validation,target_test,mixed_train,mixed_validation
label,,,,,
arb_Arab,0,0,0,3034,337
ben_Beng,0,0,0,988,110
deu_Latn,0,0,0,161,18
eng_Latn,0,0,0,2681,298
fra_Latn,0,0,0,959,107
hin_Deva,0,0,0,724,80
pli_Sinh,23490,2800,3027,23486,2799
san_Deva,0,0,0,4500,500
san_Sinh,10120,1291,1327,10119,1291


commonlid 93428 selected rows; index: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\new_method\results\main_seed42\evaluation_index_commonlid.csv
flores_plus 16155 selected rows; index: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\new_method\results\main_seed42\evaluation_index_flores_plus.csv
wili_2018 15016 selected rows; index: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\new_method\results\main_seed42\evaluation_index_wili_2018.csv
Data audit passed. Continue with notebooks 01–03.


Existing zero-shot per-example predictions may be aligned with this evaluation index and imported through `existing_zero_predictions`. Do not make up hashes or IDs for mismatched texts. If you only retained aggregate F1, rerun zero-shot or separately establish exact protocol equivalence.